In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb
import pickle
import os
import logging

# --- Configure Logging for Tuning Script ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('hyperparameter_tuning.log'),
        logging.StreamHandler()  # Also print to console
    ]
)
logger = logging.getLogger(__name__)

# --- Configuration ---
train_file = 'train_processed_final.csv'  # Path to preprocessed training data
# Use a small validation split from training data for tuning
VAL_SIZE = 0.2
N_SPLITS_CV = 3 # Reduce CV splits for faster tuning during initial runs, increase later if needed
RANDOM_STATE = 42
tuning_results_dir = 'tuning_results'
os.makedirs(tuning_results_dir, exist_ok=True)

# --- Load Data ---
logger.info("Starting hyperparameter tuning script.")
logger.info("Loading data...")
try:
    if os.path.exists(train_file):
        df_train = pd.read_csv(train_file)
        logger.info(f"Loaded training  {df_train.shape}")
    else:
        logger.error(f"Error: {train_file} not found. Please ensure preprocessed data is available.")
        exit()

except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
    exit()

# --- Prepare Features and Target ---
# Identify the target column as the LAST column in the training set
target_column = df_train.columns[-1]  # Get the name of the last column
logger.info(f"Identified target column: '{target_column}'")

if target_column not in df_train.columns:
    logger.error(f"Error: Target column '{target_column}' (assumed to be the last column) not found in training data.")
    logger.error(f"Available columns: {list(df_train.columns)}")
    exit()

X = df_train.drop(columns=[target_column])
y = df_train[target_column]

logger.info(f"Features (X) shape: {X.shape}")
logger.info(f"Target (y) shape: {y.shape}")

# --- Split for Tuning (use a small validation set) ---
X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
    X, y, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y # Stratify for classification
)
logger.info(f"Tuning - Train shape: {X_train_tune.shape}, Val shape: {X_val_tune.shape}")

# --- Define Models and Parameter Grids ---
# Using RandomizedSearchCV for models with larger parameter spaces
# Using GridSearchCV for simpler models
model_param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        },
        'search_type': 'randomized', # Use RandomizedSearchCV
        'n_iter': 50 # Number of parameter settings sampled for RandomizedSearchCV
    },
    'XGB': {
        'model': xgb.XGBClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 6, 10],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        },
        'search_type': 'randomized',
        'n_iter': 50
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 9, 11],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan', 'minkowski']
        },
        'search_type': 'grid'
    }
    # Note: GaussianNB typically has few hyperparameters to tune effectively.
    # Its default settings are often good. We could add alpha for smoothing if needed.
    # For now, it's omitted from tuning, but could be added like this:
    # 'NaiveBayes': {
    #     'model': GaussianNB(),
    #     'params': {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]},
    #     'search_type': 'grid'
    # }
}

# --- Perform Hyperparameter Tuning ---
best_models = {}
tuning_results = {}

for name, config in model_param_grids.items():
    logger.info(f"--- Tuning {name} ---")
    model = config['model']
    param_grid = config['params']
    search_type = config['search_type']
    n_iter = config.get('n_iter', None) # Get n_iter if specified, else None

    # Define the search object
    if search_type == 'grid':
        search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE),
            scoring='f1_macro', # Use F1 macro for multiclass as a good general metric
            n_jobs=-1, # Use all available cores
            verbose=1  # Print progress
        )
    elif search_type == 'randomized':
        if n_iter is None:
            logger.error(f"n_iter must be specified for RandomizedSearchCV for {name}")
            continue
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE),
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1,
            random_state=RANDOM_STATE
        )
    else:
        logger.error(f"Unknown search type '{search_type}' for {name}")
        continue

    try:
        logger.info(f"  Fitting {search_type} for {name}...")
        search.fit(X_train_tune, y_train_tune)

        best_model = search.best_estimator_
        best_params = search.best_params_
        best_score = search.best_score_

        logger.info(f"  Best params for {name}: {best_params}")
        logger.info(f"  Best cross-validation F1 macro score: {best_score:.4f}")

        # Evaluate the best model on the validation set
        y_pred_val = best_model.predict(X_val_tune)
        val_accuracy = accuracy_score(y_val_tune, y_pred_val)
        val_f1 = f1_score(y_val_tune, y_pred_val, average='macro')
        try:
            val_roc_auc = roc_auc_score(y_val_tune, best_model.predict_proba(X_val_tune), average='macro', multi_class='ovr')
        except Exception as e: # Handle potential errors in ROC AUC calculation (e.g., insufficient classes in fold)
             logger.warning(f"Could not calculate ROC AUC on validation set for {name}: {e}")
             val_roc_auc = float('nan')

        logger.info(f"  Validation Accuracy: {val_accuracy:.4f}")
        logger.info(f"  Validation F1 macro: {val_f1:.4f}")
        logger.info(f"  Validation ROC AUC (macro, ovr): {val_roc_auc:.4f}")


        # Store results
        best_models[name] = best_model
        tuning_results[name] = {
            'best_params': best_params,
            'cv_f1_macro_mean': best_score,
            'val_accuracy': val_accuracy,
            'val_f1_macro': val_f1,
            'val_roc_auc_macro': val_roc_auc
        }

        # Save the best model using pickle
        model_save_path = os.path.join(tuning_results_dir, f'best_model_{name.lower()}.pkl')
        with open(model_save_path, 'wb') as f:
            pickle.dump(best_model, f)
        logger.info(f"  Best model for {name} saved to {model_save_path}")

    except Exception as e:
        logger.error(f"  Error tuning {name}: {e}")

# --- Save Tuning Results Summary ---
results_summary_path = os.path.join(tuning_results_dir, 'tuning_results_summary.txt')
with open(results_summary_path, 'w') as f:
    for name, results in tuning_results.items():
        f.write(f"--- Model: {name} ---\n")
        f.write(f"Best Parameters: {results['best_params']}\n")
        f.write(f"Best CV F1 Macro Mean: {results['cv_f1_macro_mean']:.4f}\n")
        f.write(f"Validation Accuracy: {results['val_accuracy']:.4f}\n")
        f.write(f"Validation F1 Macro: {results['val_f1_macro']:.4f}\n")
        f.write(f"Validation ROC AUC Macro: {results['val_roc_auc_macro']:.4f}\n")
        f.write("\n")
logger.info(f"Tuning results summary saved to {results_summary_path}")

logger.info("Hyperparameter tuning completed. Check the 'tuning_results' folder for best models and summary.")

2025-11-26 17:04:24,542 - INFO - Starting hyperparameter tuning script.
2025-11-26 17:04:24,543 - INFO - Loading data...
2025-11-26 17:04:24,569 - INFO - Loaded training  (14396, 15)
2025-11-26 17:04:24,570 - INFO - Identified target column: 'Class'
2025-11-26 17:04:24,572 - INFO - Features (X) shape: (14396, 14)
2025-11-26 17:04:24,572 - INFO - Target (y) shape: (14396,)
2025-11-26 17:04:24,580 - INFO - Tuning - Train shape: (11516, 14), Val shape: (2880, 14)
2025-11-26 17:04:24,581 - INFO - --- Tuning RandomForest ---
2025-11-26 17:04:24,582 - INFO -   Fitting randomized for RandomForest...


Fitting 3 folds for each of 50 candidates, totalling 150 fits


2025-11-26 17:05:14,953 - INFO -   Best params for RandomForest: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}
2025-11-26 17:05:14,954 - INFO -   Best cross-validation F1 macro score: 0.4184
2025-11-26 17:05:15,040 - INFO -   Validation Accuracy: 0.4552
2025-11-26 17:05:15,040 - INFO -   Validation F1 macro: 0.4225
2025-11-26 17:05:15,041 - INFO -   Validation ROC AUC (macro, ovr): 0.8320
2025-11-26 17:05:15,091 - INFO -   Best model for RandomForest saved to tuning_results\best_model_randomforest.pkl
2025-11-26 17:05:15,092 - INFO - --- Tuning XGB ---
2025-11-26 17:05:15,093 - INFO -   Fitting randomized for XGB...


Fitting 3 folds for each of 50 candidates, totalling 150 fits


2025-11-26 17:06:10,525 - INFO -   Best params for XGB: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
2025-11-26 17:06:10,526 - INFO -   Best cross-validation F1 macro score: 0.4580
2025-11-26 17:06:10,554 - INFO -   Validation Accuracy: 0.4847
2025-11-26 17:06:10,555 - INFO -   Validation F1 macro: 0.4608
2025-11-26 17:06:10,556 - INFO -   Validation ROC AUC (macro, ovr): 0.8718
2025-11-26 17:06:10,600 - INFO -   Best model for XGB saved to tuning_results\best_model_xgb.pkl
2025-11-26 17:06:10,601 - INFO - --- Tuning KNN ---
2025-11-26 17:06:10,601 - INFO -   Fitting grid for KNN...


Fitting 3 folds for each of 30 candidates, totalling 90 fits


2025-11-26 17:06:15,392 - INFO -   Best params for KNN: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'uniform'}
2025-11-26 17:06:15,393 - INFO -   Best cross-validation F1 macro score: 0.3912
2025-11-26 17:06:16,235 - INFO -   Validation Accuracy: 0.4160
2025-11-26 17:06:16,236 - INFO -   Validation F1 macro: 0.3895
2025-11-26 17:06:16,236 - INFO -   Validation ROC AUC (macro, ovr): 0.7890
2025-11-26 17:06:16,242 - INFO -   Best model for KNN saved to tuning_results\best_model_knn.pkl
2025-11-26 17:06:16,243 - INFO - Tuning results summary saved to tuning_results\tuning_results_summary.txt
2025-11-26 17:06:16,244 - INFO - Hyperparameter tuning completed. Check the 'tuning_results' folder for best models and summary.
